# NIH metrics

This notebook uses information extracted from [NIH Exporter](https://reporter.nih.gov/exporter) to identify NIH-NHLBI funded users of PhysioNet.

## Import packages

In [1]:
import os
from pathlib import Path

import pandas as pd

from twentyfiveyears.nih import (combine_exporter_tables, get_physionet_users, get_investigators, get_authors, link_users)

## Setup

In [2]:
# Set the base path
base_path = os.path.join("..", "data")

## Load map of Person IDs

All users are assigned a unique `person_id`.

In [3]:
# Load the map of Person IDs
path = os.path.join(base_path, 'handcrafted', 'person_id_lookup.csv')
person_map = pd.read_csv(path)
person_map.head(3)

,person_id,physionet_id
0,100000000,2
1,100000001,6
2,100000002,8


## Load PhysioNet dataset

Load a dataset containing the list of PhysioNet users

In [4]:
# Load DataFrame of PhysioNet users
path = os.path.join(base_path, 'physionet', 'users.csv')
df_physionet_users = get_physionet_users(path, person_map, first_name_as_initial=True)
df_physionet_users.head(3)

,person_id,physionet_name
0,100000000,f torres fábregas
1,100000001,t pollard
2,100000002,b moody


## Load Principal Investigators of NIH projects

Load a list of Principal Investigators

In [5]:
# Load the NIH project data
path = os.path.join(base_path, 'nih', 'exporter', 'projects')
df_projects = combine_exporter_tables(path, "RePORTER_PRJ_C_FY", start_year=1995)

### Limit the data to NIBIB projects

In [6]:
df_projects = df_projects[df_projects['IC_NAME']=='National Heart, Lung, and Blood Institute'.upper()]
df_projects.head(3)

,APPLICATION_ID,ACTIVITY,ADMINISTERING_IC,APPLICATION_TYPE,ARRA_FUNDED,AWARD_NOTICE_DATE,BUDGET_START,BUDGET_END,CFDA_CODE,CORE_PROJECT_NUM,...,SUBPROJECT_ID,SUFFIX,SUPPORT_YEAR,TOTAL_COST,TOTAL_COST_SUB_PROJECT,OPPORTUNITY NUMBER,FUNDING_MECHANISM,ORG_IPF_CODE,DIRECT_COST_AMT,INDIRECT_COST_AMT
1995,2213630,F31,HL,5,NaN,1994-12-16T00:00:00,01/20/1995,01/19/1996,837,F31HL008814,...,NaN,NaN,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1996,2213634,F31,HL,5,NaN,1995-08-20T00:00:00,08/31/1995,08/30/1996,837,F31HL008815,...,NaN,NaN,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1997,2213969,F31,HL,5,NaN,1994-11-30T00:00:00,11/01/1994,10/31/1995,838,F31HL009056,...,NaN,NaN,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
# Get the names of Principal Investigators
investigators = get_investigators(df_projects, first_name_as_initial=True)
investigators[0:3]

['w carrasco', 't epps', 'j santiago']

## Load authors of publications linked to NIH projects

Load a list of authors linked to NIH projects

In [8]:
# Import the linking tables to connect publications / authors to a specific NIH institute
# NOTE: have to manually rename years 2016 - 2020 as RePORTER instead of REPORTER
path = os.path.join(base_path, 'nih', 'exporter', 'link_tables')
df_links = combine_exporter_tables(path, "RePORTER_PUBLNK_C_", start_year=1995)

In [9]:
# Load the NIH publications data
path = os.path.join(base_path, 'nih', 'exporter', 'publications')
df_publications = combine_exporter_tables(path, "RePORTER_PUB_C_", start_year=1995)

### Limit the data to NIBIB publications

In [10]:
# First get the PROJECT_NUM from the df_links table
df_publications = pd.merge(df_publications, df_links, on='PMID')
# Next get merge with the projects DataFrame to get 'IC_NAME'
df_publications = pd.merge(df_publications, df_projects[['CORE_PROJECT_NUM', 'IC_NAME']].copy(), left_on='PROJECT_NUMBER', right_on='CORE_PROJECT_NUM')

In [11]:
# Get a DataFrame only for NIBIB publications
# NOTE: this line isn't doing anything since we already filter the projects on this above and then merge on the associated CORE_PROJECT_NUM
df_publications = df_publications[df_publications['IC_NAME']=='National Heart, Lung, and Blood Institute'.upper()]
df_publications.head(3)

,AFFILIATION,AUTHOR_LIST,COUNTRY,ISSN,JOURNAL_ISSUE,JOURNAL_TITLE,JOURNAL_TITLE_ABBR,JOURNAL_VOLUME,LANG,PAGE_NUMBER,PMC_ID,PMID,PUB_DATE,PUB_TITLE,PUB_YEAR,PROJECT_NUMBER,CORE_PROJECT_NUM,IC_NAME
0,"Department of Medicine, College of Physicians ...","Iwami, G; Akanuma, M; Kawabe, J; Cannon, P J; ...",IRELAND,0303-7207,1-2,Molecular and cellular endocrinology.,Mol Cell Endocrinol,110,eng,43-7,NaN,7672452,1995 Apr 28,Multiplicity in type V adenylylcyclase: type V...,1995,P01HL038070,P01HL038070,"NATIONAL HEART, LUNG, AND BLOOD INSTITUTE"
1,"Department of Medicine, College of Physicians ...","Iwami, G; Akanuma, M; Kawabe, J; Cannon, P J; ...",IRELAND,0303-7207,1-2,Molecular and cellular endocrinology.,Mol Cell Endocrinol,110,eng,43-7,NaN,7672452,1995 Apr 28,Multiplicity in type V adenylylcyclase: type V...,1995,P01HL038070,P01HL038070,"NATIONAL HEART, LUNG, AND BLOOD INSTITUTE"
2,"Department of Medicine, College of Physicians ...","Iwami, G; Akanuma, M; Kawabe, J; Cannon, P J; ...",IRELAND,0303-7207,1-2,Molecular and cellular endocrinology.,Mol Cell Endocrinol,110,eng,43-7,NaN,7672452,1995 Apr 28,Multiplicity in type V adenylylcyclase: type V...,1995,P01HL038070,P01HL038070,"NATIONAL HEART, LUNG, AND BLOOD INSTITUTE"


since there are multiple publications per project and multiple project rows per PMID we get duplicate rows


In [12]:
# At this point all remaining publications are NHLBI funded so we can drop duplicated publications
df_publications = df_publications.drop_duplicates(subset="PMID")

In [18]:
# Get the names of authors
authors = get_authors(df_publications, first_name_as_initial=True)
authors[0:3]

['g iwami', 'm akanuma', 'j kawabe']

## Match NIH listed people to PhysioNet users

Attempt to match people between the two sources

In [19]:
# Match NIH Principal Investigators to PhysioNet users
# Set limit for testing
limit = None
df_physionet_users = link_users(df_physionet_users, investigators, match_group="investigators", limit=limit)

100%|██████████| 9/9 [00:00<00:00, 2516.08it/s]

Finished in 0.06467914581298828 seconds


In [20]:
# Match NIH authors to PhysioNet users
df_physionet_users = link_users(df_physionet_users, authors, match_group="authors", limit=limit)

100%|██████████| 9/9 [00:00<00:00, 2639.22it/s]


Finished in 0.4916810989379883 seconds


In [21]:
df_physionet_users.head(5)

,person_id,physionet_name,matched_investigator_score,matched_investigator_name,matched_author_score,matched_author_name
0,100000000,f torres fábregas,0.810196,f forsberg,0.894118,f torres
1,100000001,t pollard,0.925926,a pollard,1.000000,t pollard
2,100000002,b moody,0.904762,b moorthy,1.000000,b moody
3,100000003,a johnson,1.000000,a johnson,1.000000,a johnson
4,100000004,j ishii-rousseau,0.837037,j rouleau,0.862500,j ishida


Save the results

## Save the results

In [17]:
# Save the results
save_path = os.path.join(base_path, 'physionet_users_nih_nhlbi_funded.csv')
path = Path(save_path)

# Convert to a path that works on the current OS
normalized_path = path.as_posix() if path.drive else Path(*path.parts).resolve()

# Output the merged DataFrame or save it to a file
df_physionet_users.to_csv(normalized_path, index=False)